# OpenPlaque Bayesian Boundary-Parameter Tuning

This notebook runs from a clean Colab environment. It:

1. clones/pulls OpenPlaque from GitHub,
2. downloads or uses the labeled sample dataset,
3. restores cached nnU-Net predictions from Google Drive if available,
4. otherwise runs nnU-Net once per labeled case and saves a compressed prediction cache to Drive,
5. uses Optuna Bayesian optimization to search downstream boundary-refinement parameters on **all labeled data**.

No cross-validation is used. nnU-Net predictions are fixed and cached; optimization searches only post-processing parameters.

Research use only. Not clinically validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update OpenPlaque from GitHub.
from pathlib import Path
import sys, os

repo = Path('/content/OpenPlaque')
if not repo.exists():
    !git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

src = repo / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print('Using OpenPlaque source:', src)

In [ ]:
# Install dependencies, including Optuna for Bayesian optimization.
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q optuna pandas scipy matplotlib SimpleITK pydicom gdown

# GPU check. If this fails, switch Runtime > Change runtime type > GPU.
!nvidia-smi

In [ ]:
# Configure nnU-Net environment.
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Verify repository has the Bayesian tuning functions.
from openplaque.boundary_parameter_tuning import (
    BAYESIAN_SEARCH_SPACE,
    collect_sample_cases,
    detect_available_model_folds,
    ensure_prediction_cache_with_archive,
    bayesian_optimize_boundary_parameters,
    select_best_bayesian_params,
    save_bayesian_optimization_outputs,
)

print('Loaded Bayesian tuning module successfully.')

In [ ]:
# Extract nnU-Net model weights if needed.
from pathlib import Path
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing model ZIP: {model_zip}')
    with zipfile.ZipFile(model_zip) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

!find /content/nnUNet_results -maxdepth 4 | head -80

In [ ]:
# Download or locate labeled sample dataset.
# Expected pairing:
#   P02_LAD_axial_0000.nii.gz  -> CT image
#   P02_LAD_axial.nii.gz       -> expert label mask 0/1/2

from pathlib import Path

SAMPLE_DRIVE_FOLDER = 'https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing'
DATASET_CANDIDATES = [
    Path('/content/sample_dataset/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM'),
]

DATASET_DIR = next((p for p in DATASET_CANDIDATES if p.exists()), None)

if DATASET_DIR is None:
    print('Downloading sample dataset from Google Drive...')
    !rm -rf /content/sample_dataset
    !gdown --folder "https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing" -O /content/sample_dataset --remaining-ok
    DATASET_DIR = Path('/content/sample_dataset/Sample_Dataset')

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Could not find sample dataset: {DATASET_DIR}')

cases = collect_sample_cases(DATASET_DIR)
print('Using dataset:', DATASET_DIR)
print('Paired cases found:', len(cases))
print('First 10 cases:', [c.case_id for c in cases[:10]])

In [ ]:
# Display Bayesian search space.
import pandas as pd

space_rows = []
for name, spec in BAYESIAN_SEARCH_SPACE.items():
    if spec['type'] == 'categorical':
        values = spec['choices']
    else:
        values = f"{spec['type']}({spec['low']}..{spec['high']})"
    space_rows.append({'parameter': name, 'search': values})

search_space_df = pd.DataFrame(space_rows)
display(search_space_df)

print('Optimizer: Optuna TPESampler Bayesian/sequential model-based optimization')
print('Default trials below: 250. Increase N_TRIALS after confirming the workflow.')

In [ ]:
# Ensure cached predictions.
# First run: nnU-Net runs once per case, then writes a compressed archive to Drive.
# Later clean runs: predictions are restored from the Drive archive and nnU-Net is skipped.

from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning_Bayesian')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_DIR = Path('/content/openplaque_predictions')
INPUT_DIR = Path('/content/openplaque_prediction_inputs')
PRED_ARCHIVE = OUTPUT_DIR / 'nnunet_prediction_cache.zip'

folds = detect_available_model_folds('/content/nnUNet_results', dataset_name='Dataset001_CCTA_DHM')
print('Using nnU-Net folds:', folds)

ensure_prediction_cache_with_archive(
    cases=cases,
    pred_dir=PRED_DIR,
    input_dir=INPUT_DIR,
    archive_path=PRED_ARCHIVE,
    dataset_name_or_id='Dataset001_CCTA_DHM',
    configuration='3d_fullres',
    folds=folds,
    overwrite_predictions=False,
    overwrite_archive=False,
)

print('Prediction cache directory:', PRED_DIR)
print('Prediction cache archive:', PRED_ARCHIVE)

In [ ]:
# Run Bayesian optimization on all labeled sample cases.
# Increase N_TRIALS to 500-1000 for a more thorough search.

N_TRIALS = 250

study, bayesian_case_results, bayesian_trial_summary = bayesian_optimize_boundary_parameters(
    cases=cases,
    pred_dir=PRED_DIR,
    n_trials=N_TRIALS,
    seed=42,
    show_progress_bar=True,
)

final_params = select_best_bayesian_params(bayesian_trial_summary)

print('Best mean score:', study.best_value)
print('Best parameters:')
for k, v in final_params.items():
    print(f'  {k}: {v}')

display(bayesian_trial_summary.head(20))

In [ ]:
# Save outputs to Google Drive.
paths = save_bayesian_optimization_outputs(
    trial_case_results=bayesian_case_results,
    trial_summary=bayesian_trial_summary,
    final_params=final_params,
    output_dir=OUTPUT_DIR,
)

print('Saved outputs:')
for k, p in paths.items():
    print(f'  {k}: {p}')

In [ ]:
# Quick sensitivity plots for top trials.
import matplotlib.pyplot as plt

summary = bayesian_trial_summary.copy()
plt.figure(figsize=(8, 5))
plt.plot(summary['rank'], summary['mean_score'], marker='o')
plt.xlabel('Trial rank')
plt.ylabel('Mean supervised score')
plt.title('Bayesian optimization trial ranking')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(summary['mean_abs_tpv_error_fraction'], summary['mean_dice'])
plt.xlabel('Mean absolute TPV error fraction')
plt.ylabel('Mean Dice')
plt.title('Dice vs TPV error across Bayesian trials')
plt.grid(True, alpha=0.3)
plt.show()